## Modelo: Regresion Lineal

## Objetivo:
> El objetivo de este notebook es construir un modelo de recomendacion que sugiera las categorias de productos mas relevantes para cada cliente basandose en el cluster al que fue asignado.

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [15]:
model = joblib.load("../models/lgbm_customer_classifier.pkl")

df = pd.read_csv("../data/processed/olist_clustering.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115878 entries, 0 to 115877
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   product_category_name  115878 non-null  str    
 1   payment_value          115878 non-null  float64
 2   day_moment             115878 non-null  str    
 3   day_type               115878 non-null  str    
 4   season                 115878 non-null  str    
 5   hour                   115878 non-null  int64  
 6   hour_spline_0          115878 non-null  float64
 7   hour_spline_1          115878 non-null  float64
 8   hour_spline_2          115878 non-null  float64
 9   hour_spline_3          115878 non-null  float64
 10  cluster                115878 non-null  int64  
dtypes: float64(5), int64(2), str(4)
memory usage: 9.7 MB


In [16]:
df_cluster_products = df.groupby(["cluster", "product_category_name"]).size().reset_index(name="count")
df_cluster_products

,cluster,product_category_name,count
0,0,agro_industry_and_commerce,105
1,0,air_conditioning,107
2,0,art,68
3,0,arts_and_craftmanship,9
4,0,audio,126
...,...,...,...
261,3,stationery,646
262,3,tablets_printing_image,24
263,3,telephony,1093
264,3,toys,1005


In [17]:
df_cluster_products = df_cluster_products.sort_values(["cluster", "count"], ascending=[True, False])
df_cluster_products

,cluster,product_category_name,count
7,0,bed_bath_table,4306
43,0,health_beauty,3636
65,0,sports_leisure,3301
39,0,furniture_decor,3149
15,0,computers_accessories,3018
...,...,...,...
227,3,fashion_sport,7
230,3,flowers,6
247,3,la_cuisine,4
224,3,fashion_childrens_clothes,1


In [18]:
top_5_per_cluster = df_cluster_products.groupby('cluster').head(5)
top_5_per_cluster

,cluster,product_category_name,count
7,0,bed_bath_table,4306
43,0,health_beauty,3636
65,0,sports_leisure,3301
39,0,furniture_decor,3149
15,0,computers_accessories,3018
78,1,bed_bath_table,4729
114,1,health_beauty,3724
110,1,furniture_decor,3508
135,1,sports_leisure,3300
120,1,housewares,2643


In [19]:
dict_recomendation_per_cluster = top_3_per_cluster.groupby("cluster")["product_category_name"].apply(list).to_dict()
dict_recomendation_per_cluster

{0: ['bed_bath_table',
  'health_beauty',
  'sports_leisure',
  'furniture_decor',
  'computers_accessories'],
 1: ['bed_bath_table',
  'health_beauty',
  'furniture_decor',
  'sports_leisure',
  'housewares'],
 2: ['computers_accessories',
  'watches_gifts',
  'health_beauty',
  'office_furniture',
  'auto'],
 3: ['bed_bath_table',
  'health_beauty',
  'sports_leisure',
  'computers_accessories',
  'furniture_decor']}

## Función de recomendación

In [31]:
def recommend(customer_data_list):
    # Definir las columnas EXACTAMENTE como en el entrenamiento
    col_names = ["product_category_name", "payment_value", "day_moment", "day_type", "season"]
    
    # Crear un DataFrame de una fila
    X_input = pd.DataFrame([customer_data_list], columns=col_names)
    
    # Asegurarse de que las categorías sean tipo 'category' si usaste LightGBM
    for col in ["product_category_name", "day_moment", "day_type", "season"]:
        X_input[col] = X_input[col].astype('category')
        
    cluster_id = model.predict(X_input)[0]
    recommendation = dict_recomendation_per_cluster[cluster_id]
    
    return cluster_id, recommendation

# Prueba ahora así:
print(recommend(["auto", 175.26, "night", "weekend", "Summer"]))

(np.int64(2), ['computers_accessories', 'watches_gifts', 'health_beauty', 'office_furniture', 'auto'])


In [20]:
def recommend(customer_data):

    cluster_id = model.predict(customer_data)[0]

    recommendation = dict_recomendation_per_cluster[cluster_id]

    return cluster_id, recommendation

print(recommend([["bed_bath_table", 10, "night", "weekend", "winter"]]))

c:\Users\Usuario\miniconda3\envs\env_commerce_py313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ValueError: dtype='numeric' is not compatible with arrays of bytes/strings.Convert your data to numeric values explicitly instead.